# Qualitative Comparative Analysis (fsQCA) — crisis Ejecutivo-Legislativo en el Perú

Este notebook comienza **después de la calibración** realizada en `01_preQCA_Tesis_Peru`.

**Entrada:** `qca_calibrada_peru.csv`

## ¿Qué haremos y para qué?

Se siguen los mismos seis pasos del notebook `QCA_revisado.ipynb` del profesor:

1. **Leer la base calibrada.**
2. **Evaluar necesidad**: ¿alguna condición debe estar presente cuando aparece `REMOCION`?
3. **Construir la tabla de verdad**: organizar las combinaciones observadas.
4. **Identificar configuraciones suficientes**: ¿qué combinaciones se asocian sistemáticamente con `REMOCION`?
5. **Simplificar las configuraciones** mediante minimización lógica.
6. **Evaluar la solución** mediante consistencia, cobertura y PRI.

Al final se añaden bloques **adicionales** (QCA-07 a QCA-09), claramente marcados, que no están en el ejemplo docente: revisión de contradicciones, análisis de la ausencia del resultado y robustez de umbrales.

**Parámetros (iguales a los del profesor):** necesidad ≥ 0.90; consistencia ≥ 0.80; PRI ≥ 0.75; frecuencia ≥ 1.

## Instalar la librería

En Google Colab necesitamos instalar `setqca` una vez por sesión.

In [1]:
!pip -q install setqca

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

## QCA-01: leer la base preparada

Cada fila representa un episodio de crisis. `ESCUDO`, `APOYO` y `CONFLICTO` son membresías fuzzy entre 0 y 1; `REMOCION` es 0 o 1.

**Adaptación:** el profesor lee el CSV desde GitHub; aquí se lee el archivo producido por `01_preQCA_Tesis_Peru` en la misma carpeta.

In [2]:
import pandas as pd

qca = pd.read_csv("qca_calibrada_peru.csv")

# Control añadido: membresías en [0,1] y casos únicos
assert qca["caso"].is_unique
assert qca[["ESCUDO", "APOYO", "CONFLICTO", "REMOCION"]].apply(lambda s: s.between(0, 1).all()).all()
print("Casos:", len(qca))

qca.round(3)


Casos: 10


,caso,ESCUDO,APOYO,CONFLICTO,REMOCION
0,PE01,0.172,0.357,0.022,0.0
1,PE02,0.172,0.103,0.005,1.0
2,PE03,0.041,0.972,0.109,0.0
3,PE04,0.019,0.988,0.472,0.0
4,PE05,0.019,0.972,0.176,1.0
5,PE06,0.828,0.676,0.980,0.0
6,PE07,0.828,0.273,0.972,0.0
7,PE08,0.828,0.357,1.000,1.0
8,PE09,0.019,0.016,0.800,0.0
9,PE10,0.019,0.005,0.891,1.0


## QCA-02: buscar condiciones necesarias

Una condición necesaria responde a la pregunta:

> Cuando encontramos el resultado, ¿encontramos también la condición?

Usaremos `necessity_analysis()` para evaluar las tres condiciones (y su ausencia, `~`) respecto de `REMOCION`. Tomamos 0.90 como referencia.

In [3]:
from setqca import necessity_analysis

necesidad = necessity_analysis(
    qca,
    outcome="REMOCION",
    conditions=["ESCUDO", "APOYO", "CONFLICTO"],
    consistency_threshold=0.90
)

tabla_necesidad = necesidad.to_frame()[
    ["condition", "consistency", "coverage", "necessary"]
]
tabla_necesidad.to_csv("T2_necesidad.csv", index=False)
tabla_necesidad


,condition,consistency,coverage,necessary
0,~ESCUDO,0.740329,0.419846,False
1,~APOYO,0.640763,0.485263,False
2,CONFLICTO,0.517799,0.381684,False
3,~CONFLICTO,0.482201,0.421732,False
4,APOYO,0.359237,0.304553,False
5,ESCUDO,0.259671,0.352495,False


### ¿Cómo leer el resultado?

Ninguna condición, ni su ausencia, alcanza 0,90. La más alta es **~ESCUDO (0,740)**: tres de las cuatro remociones (PPK, Vizcarra, Boluarte) ocurrieron sin bancada propia capaz de bloquear la vacancia, pero Castillo fue removido teniendo el 28% de los escaños, lo que impide hablar de necesidad.

Que ninguna condición sea necesaria no significa que carezca de importancia: podría formar parte de una combinación suficiente, lo que se examina a continuación.

## QCA-03: observar combinaciones

Con tres condiciones existen `2³ = 8` configuraciones lógicamente posibles. `build_truth_table()` organiza esas configuraciones. Los casos se ubican en una fila según estén más dentro o más fuera de cada conjunto respecto de 0.5; la consistencia de cada fila se calcula con las membresías fuzzy.

In [4]:
from setqca import build_truth_table

tabla = build_truth_table(
    qca,
    outcome="REMOCION",
    conditions=["ESCUDO", "APOYO", "CONFLICTO"],
    inclusion_cutoff=0.80,
    pri_cutoff=0.75,
    frequency_cutoff=1,
    case_id="caso"
)

tabla_df = tabla.to_frame()
tabla_df.to_csv("T3_tabla_verdad.csv", index=False)
tabla_df


,ESCUDO,APOYO,CONFLICTO,minterm,n,consistency,PRI,OUT,cases,excluded_because
0,0,0,0,0,2,0.508841,0.508841,0,"PE01, PE02",consistency 0.509 below the inclusion cutoff o...
1,0,0,1,1,2,0.476024,0.476024,0,"PE09, PE10",consistency 0.476 below the inclusion cutoff o...
2,0,1,0,2,3,0.336525,0.336525,0,"PE03, PE04, PE05",consistency 0.337 below the inclusion cutoff o...
3,0,1,1,3,0,0.270570,0.270570,R,,frequency 0 below the cutoff of 1
4,1,0,0,4,0,0.429582,0.429582,R,,frequency 0 below the cutoff of 1
5,1,0,1,5,2,0.377333,0.377333,0,"PE07, PE08",consistency 0.377 below the inclusion cutoff o...
6,1,1,0,6,0,0.301108,0.301108,R,,frequency 0 below the cutoff of 1
7,1,1,1,7,1,0.269252,0.269252,0,PE06,consistency 0.269 below the inclusion cutoff o...


### ¿Qué observamos?

- De las 8 configuraciones posibles, **5 tienen casos** y 3 son remanentes lógicos sin casos (`011`, `100`, `110`).
- **Ninguna fila alcanza la consistencia de 0,80.** La más alta es `000` (sin escudo, sin apoyo, sin conflicto), con 0,509: ahí están PPK en 2017 (sobrevive) y PPK en 2018 (cae).
- Cuatro de las cinco filas observadas contienen **a la vez** episodios con y sin remoción (ver QCA-07).

## QCA-04: identificar configuraciones suficientes

> Cuando aparece una configuración, ¿aparece sistemáticamente el resultado?

`OUT = 1` identifica las filas que alcanzan el criterio de suficiencia (0.80). Mostramos sólo las columnas necesarias.

In [5]:
columnas = [
    "ESCUDO", "APOYO", "CONFLICTO",
    "n", "consistency", "OUT"
]

suficientes = tabla_df[
    tabla_df["OUT"].astype(str) == "1"
][columnas]

print("Filas suficientes:", len(suficientes))
suficientes


Filas suficientes: 0


,ESCUDO,APOYO,CONFLICTO,n,consistency,OUT


### ¿Qué observamos?

**No hay filas suficientes.** Ninguna combinación de las tres condiciones se asocia sistemáticamente con la remoción del presidente en estos diez episodios. En el ejemplo docente, cinco de ocho filas eran suficientes; aquí, ninguna.

## QCA-05: simplificar las configuraciones

Usamos `FSQCA` con los mismos criterios del profesor: consistencia mínima 0.80, frecuencia mínima 1 y PRI mínimo 0.75.

**Control añadido:** si ninguna fila es suficiente, `setqca` no puede minimizar y devuelve un error. En lugar de dejar que el notebook se detenga, se captura el error y se informa el motivo: un resultado sin solución es un **hallazgo**, no una falla del código.

In [6]:
from setqca import FSQCA

modelo_qca = FSQCA(
    consistency=0.80,
    pri=0.75,
    frequency=1
)

try:
    resultado_qca = modelo_qca.fit(
        qca,
        outcome="REMOCION",
        conditions=["ESCUDO", "APOYO", "CONFLICTO"],
        case_id="caso"
    )
    hay_solucion = True
    print("Minimización realizada.")
except Exception as error:
    resultado_qca = None
    hay_solucion = False
    print("No hay solución minimizada:", error)


No hay solución minimizada: No truth-table row is sufficient under the chosen thresholds.


## QCA-06: evaluar la solución

La **consistencia** pregunta qué tan bien se comporta la configuración como suficiente; la **cobertura**, cuánto del resultado cubre; el **PRI**, si la relación es ambigua entre el resultado y su ausencia.

In [7]:
if hay_solucion:
    resumen = resultado_qca.summary_frame("conservative")
    resumen.to_csv("T4_solucion.csv", index=False)
else:
    resumen = pd.DataFrame([{"solution": "sin solución: ninguna fila alcanza consistencia 0.80 y PRI 0.75"}])
    resumen.to_csv("T4_solucion.csv", index=False)
resumen


,solution
0,sin solución: ninguna fila alcanza consistenci...


### ¿Cómo leer el resultado?

No existe una solución que evaluar: sin filas suficientes, la minimización no tiene insumos. Por eso no se reportan consistencia, cobertura ni PRI de una solución.

Este resultado es distinto del ejemplo docente (`MAYORIA + FORTALEZA*APOYO`), y es esperable que lo sea: se replica el **procedimiento**, no el resultado. Un análisis sin solución es un hallazgo: con estas condiciones, la vacancia en el Perú no muestra una regularidad suficiente.

---
# Bloques adicionales (no están en el notebook del profesor)

Se añaden porque el sílabo permite *"un análisis de robustez que matice el resultado principal"* y porque la guía (sección 3.11) indica que los casos que no se comportan como se esperaba deben examinarse.

## QCA-07 (adicional): examinar las filas contradictorias

Una fila es **contradictoria** cuando contiene casos con y sin el resultado. Se listan sus casos para volver a la evidencia.

In [8]:
qca["fila"] = (
    (qca["ESCUDO"] > 0.5).astype(int).astype(str)
    + (qca["APOYO"] > 0.5).astype(int).astype(str)
    + (qca["CONFLICTO"] > 0.5).astype(int).astype(str)
)
contradicciones = (qca.groupby("fila")
                   .agg(casos=("caso", ", ".join),
                        removidos=("REMOCION", "sum"),
                        n=("caso", "size"))
                   .reset_index())
contradicciones["contradictoria"] = (contradicciones["removidos"] > 0) & (contradicciones["removidos"] < contradicciones["n"])
contradicciones.to_csv("T5_contradicciones.csv", index=False)
contradicciones


,fila,casos,removidos,n,contradictoria
0,000,"PE01, PE02",1.0,2,True
1,001,"PE09, PE10",1.0,2,True
2,010,"PE03, PE04, PE05",1.0,3,True
3,101,"PE07, PE08",1.0,2,True
4,111,PE06,0.0,1,False


### ¿Qué observamos?

Todas las filas contradictorias agrupan **al mismo presidente** en dos intentos distintos:

| Fila | Sobrevive | Cae |
|---|---|---|
| `000` | PPK, dic. 2017 | PPK, mar. 2018 |
| `001` | Boluarte, abr. 2024 | Boluarte, oct. 2025 |
| `010` | Vizcarra, sep. 2019 y sep. 2020 | Vizcarra, nov. 2020 |
| `101` | Castillo, mar. 2022 | Castillo, dic. 2022 |

Las tres condiciones describen al **gobierno** (bancada, popularidad, clima social) y cambian poco entre un intento y el siguiente; el resultado, en cambio, cambia. Lo que distingue el intento fallido del exitoso está en el **episodio** y no fue medido aquí: por ejemplo, la difusión de los videos de compra de votos (PPK, 2018), las declaraciones de aspirantes a colaboradores eficaces (Vizcarra, 2020), el intento de disolver el Congreso (Castillo, 2022) o la cercanía de las elecciones de 2026 (Boluarte, 2025). Esto es una **hipótesis a examinar con rastreo de procesos**, no un resultado de QCA.

## QCA-08 (adicional): la ausencia del resultado (`~REMOCION`, supervivencia del presidente)

En QCA la causalidad es **asimétrica**: explicar por qué cae un presidente no explica automáticamente por qué sobrevive. Se repite la tabla de verdad para `NO_REMOCION = 1 − REMOCION` con los mismos criterios.

In [9]:
qca["NO_REMOCION"] = 1 - qca["REMOCION"]

tabla_no = build_truth_table(
    qca,
    outcome="NO_REMOCION",
    conditions=["ESCUDO", "APOYO", "CONFLICTO"],
    inclusion_cutoff=0.80,
    pri_cutoff=0.75,
    frequency_cutoff=1,
    case_id="caso"
).to_frame()
tabla_no.to_csv("T6_tabla_verdad_no_remocion.csv", index=False)

try:
    res_no = FSQCA(consistency=0.80, pri=0.75, frequency=1).fit(
        qca, outcome="NO_REMOCION", conditions=["ESCUDO", "APOYO", "CONFLICTO"], case_id="caso")
    sol_no = res_no.summary_frame("conservative")
except Exception as error:
    sol_no = pd.DataFrame([{"solution": f"sin solución: {error}"}])
sol_no.to_csv("T6b_solucion_no_remocion.csv", index=False)
display(tabla_no)
sol_no


,ESCUDO,APOYO,CONFLICTO,minterm,n,consistency,PRI,OUT,cases,excluded_because
0,0,0,0,0,2,0.491159,0.491159,0,"PE01, PE02",consistency 0.491 below the inclusion cutoff o...
1,0,0,1,1,2,0.523976,0.523976,0,"PE09, PE10",consistency 0.524 below the inclusion cutoff o...
2,0,1,0,2,3,0.663475,0.663475,0,"PE03, PE04, PE05",consistency 0.663 below the inclusion cutoff o...
3,0,1,1,3,0,0.729430,0.729430,R,,frequency 0 below the cutoff of 1
4,1,0,0,4,0,0.570418,0.570418,R,,frequency 0 below the cutoff of 1
5,1,0,1,5,2,0.622667,0.622667,0,"PE07, PE08",consistency 0.623 below the inclusion cutoff o...
6,1,1,0,6,0,0.698892,0.698892,R,,frequency 0 below the cutoff of 1
7,1,1,1,7,1,0.730748,0.730748,0,PE06,consistency 0.731 below the inclusion cutoff o...


,solution
0,sin solución: No truth-table row is sufficient...


### ¿Qué observamos?

Tampoco hay configuraciones suficientes para la supervivencia del presidente. La fila más consistente es `111` (escudo, apoyo y conflicto: Castillo en diciembre de 2021), con 0,731, por debajo del corte y con un solo caso.

## QCA-09 (adicional): robustez de los umbrales

Se repite el análisis con umbrales alternativos **fijados antes de ver sus resultados**, para comprobar que la conclusión no depende de una elección particular. Se descarta cualquier especificación que deje un caso exactamente en 0.5.

In [10]:
from setqca import calibrate_direct
raw = pd.read_csv("vacancias_peru.csv")
alternativas = {
    "Base":                         {"ESCUDO": (5, 20, 35), "APOYO": (15, 30, 50), "CONFLICTO": (100, 180, 250)},
    "ESCUDO al tercio exacto":      {"ESCUDO": (5, 17, 34), "APOYO": (15, 30, 50), "CONFLICTO": (100, 180, 250)},
    "APOYO más exigente":           {"ESCUDO": (5, 20, 35), "APOYO": (20, 33, 55), "CONFLICTO": (100, 180, 250)},
    "CONFLICTO por cuartiles":      {"ESCUDO": (5, 20, 35), "APOYO": (15, 30, 50), "CONFLICTO": (136, 191, 242)},
    "CONFLICTO más exigente":       {"ESCUDO": (5, 20, 35), "APOYO": (15, 30, 50), "CONFLICTO": (150, 220, 300)},
}
medicion = {"ESCUDO": "escanos_gobierno_pct", "APOYO": "aprobacion_pct", "CONFLICTO": "protestas_mes"}
filas = []
for nombre, u in alternativas.items():
    d = pd.DataFrame({"caso": raw["caso"], "REMOCION": raw["remocion"].astype(float)})
    for c, (fo, co, fi) in u.items():
        d[c] = calibrate_direct(raw[medicion[c]], full_out=fo, crossover=co, full_in=fi)
    if (d[list(u)].round(6) == 0.5).any().any():
        filas.append({"especificacion": nombre, "resultado": "descartada: caso en 0.5"}); continue
    d["NO_REMOCION"] = 1 - d["REMOCION"]
    for out in ["REMOCION", "NO_REMOCION"]:
        tt = build_truth_table(d, outcome=out, conditions=["ESCUDO", "APOYO", "CONFLICTO"], inclusion_cutoff=0.80,
                               pri_cutoff=0.75, frequency_cutoff=1, case_id="caso").to_frame()
        pos = tt[tt["OUT"].astype(str) == "1"]
        try:
            sol = FSQCA(consistency=0.80, pri=0.75, frequency=1).fit(
                d, outcome=out, conditions=["ESCUDO", "APOYO", "CONFLICTO"], case_id="caso").summary_frame("conservative")
            texto = sol["solution"].iloc[0] + f" (cons. {sol['consistency'].iloc[0]:.3f}, cob. {sol['coverage'].iloc[0]:.3f})"
        except Exception:
            texto = "sin solución"
        filas.append({"especificacion": nombre, "resultado": out,
                       "filas_suficientes": len(pos), "solucion_conservadora": texto})
robustez = pd.DataFrame(filas)
robustez.to_csv("T7_robustez.csv", index=False)
robustez


,especificacion,resultado,filas_suficientes,solucion_conservadora
0,Base,REMOCION,0,sin solución
1,Base,NO_REMOCION,0,sin solución
2,ESCUDO al tercio exacto,REMOCION,0,sin solución
3,ESCUDO al tercio exacto,NO_REMOCION,0,sin solución
4,APOYO más exigente,REMOCION,0,sin solución
5,APOYO más exigente,NO_REMOCION,0,sin solución
6,CONFLICTO por cuartiles,REMOCION,0,sin solución
7,CONFLICTO por cuartiles,NO_REMOCION,0,sin solución
8,CONFLICTO más exigente,REMOCION,0,sin solución
9,CONFLICTO más exigente,NO_REMOCION,0,sin solución


### ¿Qué observamos?

En las cinco especificaciones de umbrales, **ninguna fila es suficiente** ni para la remoción ni para la supervivencia. El resultado no es un artefacto de la elección de umbrales: se mantiene al anclar el escudo en el tercio exacto, al endurecer el umbral de aprobación y al calibrar el conflicto con los cuartiles de la serie o con umbrales más exigentes.

## Síntesis

**Interpretación.** Ninguna de las tres condiciones —escudo legislativo, apoyo ciudadano y conflictividad social— resultó necesaria para la remoción del presidente: la más cercana, la ausencia de escudo, alcanzó 0,740 frente al criterio de 0,90. Ninguna de las cinco configuraciones observadas fue suficiente: la más consistente llegó a 0,509. Cuatro de esas cinco configuraciones son contradictorias y, en todos los casos, la contradicción enfrenta dos intentos contra un mismo presidente. El resultado se mantiene con cinco especificaciones alternativas de umbrales y también para la supervivencia del presidente.

**Conclusión.** La evidencia no respalda que la vacancia presidencial en el Perú se explique por la combinación de escudo legislativo, apoyo ciudadano y conflicto social: estos factores caracterizan a cada gobierno, pero no distinguen el intento de vacancia que fracasa del que triunfa contra el mismo presidente. Esto no significa que sean irrelevantes —describen las condiciones de vulnerabilidad en que ocurren todos los intentos— sino que no bastan. Queda pendiente identificar qué cambia entre un intento y el siguiente; los cuatro pares contradictorios (PPK, Vizcarra, Castillo, Boluarte) son los casos que la investigación cualitativa de la tesis debe reconstruir mediante rastreo de procesos.